In [ ]:
from datetime import datetime
import pandas as pd
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
%run compile_data.ipynb
%run accessDMI_metObs.ipynb

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
def Sarimax_nogap(Type,file_path,endo_name,exo_names:list,granularity=10,cut_in = 1 ,order = (0,0,0),seasonal_order = (1,1,1),save_new_file = False,filename=""):
    """Using the SARIMAX function to fill gaps in the endogenous variable using the exogeneous variable
    Granularity is the amount of time between measurements in minutes
    Type is either "Wind" or "Solar" """
    exo_name = exo_names[0]

    try:
        df = pd.read_csv(file_path,delimiter=",")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    df["ts"] = pd.to_datetime(df["ts"])
    df = df.set_index("ts") # Set datetime as index
    df = df.loc[df.index >"2025-05-14 07:00:00"] # start from when our power data starts
    df = df.asfreq(f'{granularity}min')

    for ex in exo_names:
        df[ex] = df[ex].interpolate() # REMOVE ANY NANS

    df_modelled = df.copy()
    
    if Type == "Wind":
        # Wind power is proportional to the cubic wind speed
        df.loc[df[exo_name] < cut_in, endo_name] = 0 # set low wind periods to zero

        df = df.loc[df[exo_name] >= cut_in] # remove low wind periods to reduce how much it gets skewed

        df["active"] = (df[exo_name]>= 3).astype(int)
        df["w3"] = df[exo_name]**3
        
        endo = df[endo_name]
        exo = df[[*exo_names,"active","w3"]]
        print(f"Preparing Model for Wind Turbine {endo_name}")

    elif Type == "Solar":
        df.loc[df[exo_name] < cut_in, endo_name] = 0 # set night time to zero
        
        df = df.loc[df[exo_name] >= cut_in] # remove night time to reduce how much it gets skewed

        endo = df[endo_name]
        exo = df[exo_names] 
        print(f"Preparing Model for PV unit {endo_name}")


    fraction_dayvsnight = len(df)/len(df_modelled)
    seasonality = 24*60/granularity*fraction_dayvsnight ####### Takes the average time between the time today and the same time tomorrow (complex since we removed nighttime values)
    seasonal_order_full = (*seasonal_order, int(seasonality))

    model = SARIMAX(
        endo,
        exog=exo,
        order=order,
        seasonal_order=seasonal_order_full,  # daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit()

    endo_modelled = results.get_prediction().predicted_mean # USES KALMAN SMOOTHING ( includes both past and future values to do "predictions")
    df["endo_filled"] = df[endo_name]
    df.loc[endo.isna(),"endo_filled"] = endo_modelled.loc[endo.isna()]



    # results.plot_diagnostics() # in case we want to see built in diagnostics 

    df_modelled.loc[df_modelled[exo_name] >= cut_in, endo_name +"_m"] = df["endo_filled"] # redo the zero rows that were removed because irradiance was zero
    df_modelled.loc[df_modelled[exo_name] < cut_in, endo_name+ "_m"] = 0 # set everything at night to 0
            
    if save_new_file:
        file_name = f'{filename}.csv'
        df_modelled = df_modelled[endo_name +"_m"]
        df_modelled.to_csv(str(ROOT) + f"/data/{file_name}", index=True)
        print(f"{filename}.csv saved to files.")
    return 



In [ ]:
raw_data = str(ROOT / "data/combined_data_DMI_gen_2_30min.csv")
PV_params = ["radia_glob","temp_dry"]
WS_params = ["wind_speed","wind_max"]
resolution = 30 # minutes 
PV_order = (0,0,0)
PV_seasonal_order = (1,0,1)
WT_order = (0,0,0)
WT_seasonal_order = (1,1,1)

In [ ]:
WTS = ["Gaia_WT Power","Aircon_WT Power"]
buildings = ["B715","B117","B319","B330_1","B330_2","B330_3","B716"]
for building in buildings: 
    Sarimax_nogap("Solar",raw_data,building,PV_params,granularity = resolution,order = PV_order, cut_in= -1, # nocutin
                                seasonal_order = PV_seasonal_order,save_new_file=True, filename=f"PV{building}_modelled_{resolution}min")
for WT in WTS:
    Sarimax_nogap("Wind",raw_data,WT,WS_params,granularity = resolution,order = WT_order, cut_in= -1, #nocutin
                                seasonal_order = WT_seasonal_order,save_new_file=True, filename=f"{WT[:-9]}_modelled_{resolution}min")

## This line took me about 10 min - for your reference ;)

In [ ]:
files = [str(ROOT) + "/data/PVB117_modelled_30min.csv",
         str(ROOT) + "/data/PVB319_modelled_30min.csv",
         str(ROOT) + "/data/PVB330_1_modelled_30min.csv",
         str(ROOT) + "/data/PVB330_2_modelled_30min.csv",
         str(ROOT) + "/data/PVB330_3_modelled_30min.csv",
         str(ROOT) + "/data/PVB716_modelled_30min.csv",
         str(ROOT) + "/data/PVB715_modelled_30min.csv",
         str(ROOT) + "/data/Aircon_modelled_30min.csv",
         str(ROOT) + "/data/Gaia_modelled_30min.csv",
         ROOT / "data/DMI_windspeed.csv", ## EMI WILL HAVE TO RUN THE accessDMI_metObs file (should be done at the beginning of this doc)
         ROOT / "data/DMI_radiation.csv" ## EMI WILL HAVE TO RUN THE accessDMI_metObs file (should be done at the beginning of this doc)
         ]

In [ ]:
def combine_final_data(files:list,filename,save_new_file=True,resolution = False):
    df_combined = pd.read_csv(files[0], 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  # Your timestamp column name
                     index_col='ts')
    
    for file in files[1:]: 
        df_temp = pd.read_csv(file, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  # Your timestamp column name
                     index_col='ts')
        if df_temp.index[1]-df_temp.index[0] != pd.Timedelta(minutes=resolution):
            df_temp = df_temp.resample(f'{resolution}min').mean() # resize the resolution

        df_temp = df_temp.loc[df_temp.index > "2025-05-14 07:00:00"]
        
        df_combined = df_combined.join(df_temp,how="outer") ## outer = keep ALL timestamps included in ALL the files, inner, left and right still exist! (decide perhaps for inner later on)
        print(f"Added {file}, shape is now: {df_combined.shape}")


    if save_new_file == False: 
        return df_combined

    df_combined.to_csv(ROOT / f'data/{filename}_{resolution}min.csv', index=True, sep =",") ## Save to one folder up and then into data
    print(f"\nFinal combined shape: {df_combined.shape}")
    print(f"Columns: {list(df_combined.columns)}")

In [ ]:
combine_final_data(files=files,filename = "complete_dataframe",save_new_file=True,resolution = 30)